# Characterize the Complex Particle Choi-Transfer Gap

Loads the completed `N20`, `Ny={30,40,50}`, `cycles=2Ny` particle-gap campaign and produces the gap-versus-cycle and terminal finite-size plots. Each saved figure is generated in its own directly editable cell.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_regularized_choi_transfer_matrix'
CAMPAIGN_ID = 'N20_Ny30-50_nsh1_a1-1_S25_cycles2Ny'
# Set this to an absolute Drive path if the bundle is stored under a custom parent.
BUNDLE_ROOT_OVERRIDE = None  # e.g. '/content/drive/MyDrive/my_project/colab_regularized_choi_transfer_matrix'
CAMPAIGN_RELATIVE = Path('gpu_data') / 'complex_particle_choi_transfer' / 'campaigns' / CAMPAIGN_ID
def has_completed_campaign(path: Path) -> bool:
    return (path / CAMPAIGN_RELATIVE / 'run_index.csv').exists()

def find_bundle_root() -> Path:
    candidates = []
    if BUNDLE_ROOT_OVERRIDE is not None:
        candidates.append(Path(BUNDLE_ROOT_OVERRIDE).expanduser())
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend((base, base / BUNDLE_NAME))
    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        candidates.extend((drive_root / BUNDLE_NAME,
                           drive_root / 'class_A_fermionic_adaptive_circuit' / BUNDLE_NAME))
        candidates.extend(p for p in drive_root.rglob(BUNDLE_NAME) if p.is_dir())
    checked = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if has_completed_campaign(candidate):
            return candidate
    raise FileNotFoundError(
        f'Could not locate completed campaign {CAMPAIGN_ID}. Characterization requires '
        'gpu_data/complex_particle_choi_transfer/campaigns/<campaign_id>/run_index.csv. '
        'Run the generation notebook first or set BUNDLE_ROOT_OVERRIDE to the bundle containing the data.'
    )

BUNDLE_ROOT = find_bundle_root()
CAMPAIGN_ROOT = BUNDLE_ROOT / CAMPAIGN_RELATIVE
FIG_ROOT = BUNDLE_ROOT / 'figs'
FIG_ROOT.mkdir(parents=True, exist_ok=True)
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_family = 'CMU Sans Serif' if 'CMU Sans Serif' in available_fonts else 'DejaVu Sans'
mpl.rcParams.update({'font.family': font_family, 'font.size': 8, 'axes.titlesize': 8,
                     'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
                     'legend.fontsize': 7, 'figure.dpi': 300, 'savefig.dpi': 300,
                     'axes.linewidth': 0.7, 'lines.linewidth': 1.2})
COLORS = {30: 'tab:blue', 40: 'tab:orange', 50: 'tab:green'}
print(f'campaign root: {CAMPAIGN_ROOT}')
print(f'figure root: {FIG_ROOT}, font={font_family}')


Mounted at /content/drive


In [ ]:
run_index = pd.read_csv(CAMPAIGN_ROOT / 'run_index.csv')
runs = {}
for row in run_index.to_dict(orient='records'):
    run_dir = Path(row['output_dir'])
    if not run_dir.exists():
        run_dir = CAMPAIGN_ROOT / 'runs' / row['config_id']
    summary = json.loads((run_dir / 'run_summary.json').read_text())
    with np.load(run_dir / 'particle_choi_transfer_gap_vs_cycle.npz', allow_pickle=False) as data:
        cycles = data['cycles'].copy()
        gap = data['gap'].copy()
        exponents = data['near_gap_exponents'].copy()
        stable = data['stable_at_cycle'].copy()
    with np.load(run_dir / 'particle_choi_transfer_final_spectrum.npz', allow_pickle=False) as data:
        final_spectrum = data['final_spectrum'].copy()
    assert gap.shape == (25, 2 * int(summary['Ny']))
    assert exponents.shape == (25, 2 * int(summary['Ny']), 3)
    assert final_spectrum.shape[0] == 25
    assert stable.shape == gap.shape
    assert np.isfinite(gap[stable]).all()
    assert np.isnan(gap[~stable]).all()
    runs[row['config_id']] = {'summary': summary, 'cycles': cycles, 'gap': gap, 'stable': stable, 'final_spectrum': final_spectrum}
display(run_index)
print(f'Loaded and validated {len(runs)} runs.')

def sample_mean_stderr(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    counts = np.sum(np.isfinite(values), axis=0)
    means = np.full(values.shape[1:], np.nan, dtype=np.float64)
    sems = np.full(values.shape[1:], np.nan, dtype=np.float64)
    for idx in range(values.shape[1]):
        retained = values[:, idx][np.isfinite(values[:, idx])]
        if retained.size:
            means[idx] = retained.mean()
            sems[idx] = retained.std(ddof=1) / np.sqrt(retained.size) if retained.size > 1 else 0.0
    return means, sems, counts

def select_runs(dw: bool):
    selected = [run for run in runs.values() if bool(run['summary']['DW']) == bool(dw)]
    return sorted(selected, key=lambda run: int(run['summary']['Ny']))


## Gap Versus Cycle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.35), constrained_layout=True, sharey=True)
for ax, dw, title in zip(axes, (False, True), ('DW off', 'DW on, slab-only')):
    for run in select_runs(dw):
        ny = int(run['summary']['Ny'])
        mean, sem, count = sample_mean_stderr(run['gap'])
        ax.plot(run['cycles'], mean, color=COLORS[ny], label=fr'$N_y={ny}$')
        ax.fill_between(run['cycles'], mean - sem, mean + sem, color=COLORS[ny], alpha=0.18, linewidth=0)
    ax.set_xlabel('cycle')
    ax.set_title(title)
    ax.set_yscale('log')
    ax.legend(frameon=False)
axes[0].set_ylabel(r'$\Delta_\lambda(t)=\min_j |\lambda_{p,j}(t)|$')
fig_path_pdf = FIG_ROOT / 'fig_particle_choi_gap_vs_cycle.pdf'
fig_path_png = FIG_ROOT / 'fig_particle_choi_gap_vs_cycle.png'
fig.savefig(fig_path_pdf, bbox_inches='tight')
fig.savefig(fig_path_png, bbox_inches='tight')
plt.show()
print(fig_path_pdf)


## Final Gap Versus System Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.35), constrained_layout=True, sharey=True)
for ax, dw, title in zip(axes, (False, True), ('DW off', 'DW on, slab-only')):
    nys, means, sems = [], [], []
    for run in select_runs(dw):
        ny = int(run['summary']['Ny'])
        final_gap = run['gap'][:, -1]
        nys.append(ny)
        retained = final_gap[np.isfinite(final_gap)]
        means.append(float(retained.mean()) if retained.size else np.nan)
        sems.append(float(retained.std(ddof=1) / np.sqrt(retained.size)) if retained.size > 1 else (0.0 if retained.size else np.nan))
    ax.errorbar(nys, means, yerr=sems, fmt='o-', color='tab:blue', capsize=3, ms=4)
    ax.set_xlabel(r'$N_y$')
    ax.set_xticks(nys)
    ax.set_yscale('log')
    ax.set_title(title)
axes[0].set_ylabel(r'$\Delta_\lambda(T=2N_y)$')
fig_path_pdf = FIG_ROOT / 'fig_particle_choi_gap_vs_Ny.pdf'
fig_path_png = FIG_ROOT / 'fig_particle_choi_gap_vs_Ny.png'
fig.savefig(fig_path_pdf, bbox_inches='tight')
fig.savefig(fig_path_png, bbox_inches='tight')
plt.show()
print(fig_path_pdf)


## Numerically Stable Choi Survival Fraction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.35), constrained_layout=True, sharey=True)
for ax, dw, title in zip(axes, (False, True), ('DW off', 'DW on, slab-only')):
    for run in select_runs(dw):
        ny = int(run['summary']['Ny'])
        survival = run['stable'].mean(axis=0)
        ax.plot(run['cycles'], survival, color=COLORS[ny], label=fr'$N_y={ny}$')
    ax.set_xlabel('cycle')
    ax.set_title(title)
    ax.set_ylim(-0.04, 1.04)
    ax.legend(frameon=False)
axes[0].set_ylabel(r'$f_{\mathrm{stable}}(t)$')
fig_path_pdf = FIG_ROOT / 'fig_particle_choi_stable_fraction_vs_cycle.pdf'
fig_path_png = FIG_ROOT / 'fig_particle_choi_stable_fraction_vs_cycle.png'
fig.savefig(fig_path_pdf, bbox_inches='tight')
fig.savefig(fig_path_png, bbox_inches='tight')
plt.show()
print(fig_path_pdf)


In [ ]:
from IPython.display import Javascript, display
try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
